# Clase 7 â VisiÃ³n Artificial en Plantas Pesqueras
## Curso: Inteligencia Artificial Aplicada a la ProducciÃ³n Pesquera
### UTN FRCh Â· PesquerosEnIA Â· 2026

**Docente:** Damian Adolfo Giacone  
**Notebook de apoyo:** Ariel Giamportone

---

## Objetivos de este notebook

1. Entender quÃ© es la visiÃ³n artificial y cÃ³mo se aplica en plantas de procesamiento pesquero
2. Explorar caracterÃ­sticas de imagen que permiten clasificar especies y evaluar calidad
3. Construir un clasificador con features simuladas de imagen (color, textura, morfologÃ­a)
4. Comprender la arquitectura de una red neuronal convolucional (CNN) sin necesidad de GPU
5. Evaluar el desempeÃ±o de un modelo de clasificaciÃ³n de calidad de producto

---

## Contexto: Â¿Por quÃ© visiÃ³n artificial en el sector pesquero?

Una planta procesadora de merluza hubbsi puede procesar **50â120 toneladas por dÃ­a**.  
El control de calidad manual implica:
- Operarios realizando inspecciÃ³n visual subjetiva
- Fatiga que aumenta el error a lo largo del turno
- Velocidad limitada (~60-80 peces/minuto en lÃ­neas rÃ¡pidas)
- Variabilidad entre operarios

**Con visiÃ³n artificial:**
- Velocidad: 200-400 peces/minuto (consistente)
- Objetividad: mismo criterio en todo el turno
- Trazabilidad: cada pieza registrada con su score de calidad
- IntegraciÃ³n con lÃ­nea de clasificaciÃ³n automÃ¡tica

**Proveedores ya operando en Argentina:** Marel, Baader, TriVision

## Parte 0 â Imports y configuraciÃ³n

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11
})

# Paleta del curso
UTNAZUL    = '#00467F'
UTNCELESTE = '#0099CC'
PESCATEAL  = '#059669'
TECHGOLD   = '#D4A017'
PESCAOCEAN = '#3B82F6'
ARIELBLUE  = '#2C5F7C'

print('Imports OK. Versiones:')
import sklearn; print(f'  scikit-learn: {sklearn.__version__}')
print(f'  numpy: {np.__version__}')
print(f'  pandas: {pd.__version__}')

---
## Parte 1 â Â¿QuÃ© caracterÃ­sticas extrae un sistema de visiÃ³n artificial?

Las cÃ¡maras industriales de alta velocidad capturan imÃ¡genes de cada pieza en la lÃ­nea de procesamiento.  
Los algoritmos extraen automÃ¡ticamente **features** (caracterÃ­sticas) de esas imÃ¡genes.

### Principales grupos de features en visiÃ³n pesquera:

| Grupo | Features extraÃ­das | QuÃ© detecta |
|-------|-------------------|-------------|
| **Color** | R, G, B medios; luminosidad; saturaciÃ³n | Frescura (color de la carne y piel) |
| **Textura** | Contraste, homogeneidad, energÃ­a (GLCM) | ParÃ¡sitos, defectos superficiales |
| **MorfologÃ­a** | Largo, ancho, Ã¡rea, relaciÃ³n de aspecto | Talla, especie, rendimiento |
| **Forma** | Convexidad, circularidad, perÃ­metro | DaÃ±os mecÃ¡nicos, cortes irregulares |
| **Espectral** | NIR, UV si disponible | ComposiciÃ³n (grasa, proteÃ­na) |

### 1.1 â Visualizar cÃ³mo varÃ­a el color segÃºn especie y calidad

In [ ]:
# Simulamos las caracterÃ­sticas tÃ­picas de color por especie (valores RGB normalizados 0-255)
species_color_profiles = {
    'Merluza hubbsi': {
        'R': (195, 15), 'G': (185, 12), 'B': (175, 10),  # (media, std)
        'descripcion': 'Carne blanca, textura firme, piel plateada con tonos verdosos',
        'color_hex': '#C3B9AF'
    },
    'Polaca': {
        'R': (210, 12), 'G': (200, 10), 'B': (188, 9),
        'descripcion': 'Carne muy blanca, casi sin pigmentaciÃ³n, textura mÃ¡s suave',
        'color_hex': '#D2C8BC'
    },
    'CastaÃ±eta': {
        'R': (160, 18), 'G': (140, 15), 'B': (120, 14),
        'descripcion': 'Carne mÃ¡s oscura, alta concentraciÃ³n de mioglobina, piel marrÃ³n-rojiza',
        'color_hex': '#A08C78'
    },
    'Langostino': {
        'R': (230, 20), 'G': (160, 25), 'B': (140, 22),
        'descripcion': 'Cuerpo translÃºcido crudo, torna anaranjado-rosa al cocinar',
        'color_hex': '#E6A08C'
    },
}

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
axes = axes.flatten()

for idx, (especie, props) in enumerate(species_color_profiles.items()):
    ax = axes[idx]
    # Simulamos 200 puntos de mediciÃ³n de color
    n = 200
    r = np.random.normal(props['R'][0], props['R'][1], n)
    g = np.random.normal(props['G'][0], props['G'][1], n)
    b = np.random.normal(props['B'][0], props['B'][1], n)
    
    ax.scatter(r, g, c=np.clip(np.stack([r, g, b], axis=1)/255, 0, 1),
               s=20, alpha=0.6, edgecolors='none')
    ax.set_xlabel('Canal R (rojo)')
    ax.set_ylabel('Canal G (verde)')
    ax.set_title(especie, fontweight='bold', color=UTNAZUL)
    ax.set_xlim(100, 255)
    ax.set_ylim(100, 255)
    ax.text(0.02, 0.02, props['descripcion'], transform=ax.transAxes,
            fontsize=8, color='gray', wrap=True,
            verticalalignment='bottom')

plt.suptitle('DistribuciÃ³n de color (R vs G) por especie\n'
             'Cada punto = una pieza escaneada en la lÃ­nea',
             fontsize=13, fontweight='bold', color=UTNAZUL)
plt.tight_layout()
plt.show()

print("ObservaciÃ³n: el espacio de color es discriminativo entre especies.")
print("Un clasificador puede aprender estas diferencias de forma automÃ¡tica.")

---
## Parte 2 â Dataset simulado: features extraÃ­das de 2.000 piezas

Generamos un dataset realista que representa las features extraÃ­das por un sistema de visiÃ³n  
en una lÃ­nea de procesamiento de merluza hubbsi.

In [ ]:
def generar_dataset_vision(n=2000, seed=42):
    """
    Genera dataset simulado de features extraÃ­das de imÃ¡genes de merluza.
    Cada fila = una pieza escaneada en la lÃ­nea de procesamiento.
    
    Features:
    - color_R, color_G, color_B: canales de color medios (0-255)
    - luminosidad: brillo percibido
    - saturacion: viveza del color
    - contraste_textura: variaciÃ³n local de intensidad (GLCM)
    - homogeneidad_textura: uniformidad de superficie
    - largo_cm: largo del filete en cm
    - ancho_cm: ancho del filete en cm
    - area_cm2: Ã¡rea calculada
    - convexidad: proporciÃ³n del Ã¡rea respecto al hull convexo (defectos de forma)
    - manchas_detectadas: nÃºmero de manchas o defectos detectados
    
    Target:
    - calidad: A (premium), B (estÃ¡ndar), C (descarte/sub-producto)
    """
    rng = np.random.default_rng(seed)
    
    # Proporciones de calidad en una planta tÃ­pica
    n_A = int(n * 0.35)  # 35% premium
    n_B = int(n * 0.50)  # 50% estÃ¡ndar
    n_C = n - n_A - n_B  # 15% descarte
    
    records = []
    
    for calidad, count in [('A', n_A), ('B', n_B), ('C', n_C)]:
        
        if calidad == 'A':
            # Grado A: carne muy blanca, sin manchas, tamaÃ±o uniforme
            R = rng.normal(200, 8, count)
            G = rng.normal(192, 7, count)
            B = rng.normal(185, 6, count)
            lum = rng.normal(192, 6, count)
            sat = rng.normal(0.12, 0.03, count)
            contraste = rng.normal(0.08, 0.02, count)
            homog = rng.normal(0.88, 0.04, count)
            largo = rng.normal(28, 3, count)
            ancho = rng.normal(8, 1, count)
            conv = rng.normal(0.95, 0.02, count)
            manchas = rng.poisson(0.3, count)
            
        elif calidad == 'B':
            # Grado B: algunas variaciones de color, manchas menores
            R = rng.normal(185, 15, count)
            G = rng.normal(175, 14, count)
            B = rng.normal(165, 13, count)
            lum = rng.normal(175, 12, count)
            sat = rng.normal(0.22, 0.06, count)
            contraste = rng.normal(0.18, 0.05, count)
            homog = rng.normal(0.75, 0.07, count)
            largo = rng.normal(24, 5, count)
            ancho = rng.normal(7, 1.5, count)
            conv = rng.normal(0.88, 0.04, count)
            manchas = rng.poisson(1.5, count)
            
        else:  # C
            # Grado C: manchas, decoloraciÃ³n, forma irregular
            R = rng.normal(165, 25, count)
            G = rng.normal(150, 22, count)
            B = rng.normal(135, 20, count)
            lum = rng.normal(150, 20, count)
            sat = rng.normal(0.40, 0.12, count)
            contraste = rng.normal(0.35, 0.10, count)
            homog = rng.normal(0.58, 0.10, count)
            largo = rng.normal(20, 7, count)
            ancho = rng.normal(6, 2, count)
            conv = rng.normal(0.78, 0.08, count)
            manchas = rng.poisson(4.5, count)
        
        area = largo * ancho * rng.normal(0.72, 0.05, count)  # factor de forma
        
        for i in range(count):
            records.append({
                'color_R': np.clip(R[i], 100, 255),
                'color_G': np.clip(G[i], 90, 255),
                'color_B': np.clip(B[i], 80, 255),
                'luminosidad': np.clip(lum[i], 80, 255),
                'saturacion': np.clip(sat[i], 0, 1),
                'contraste_textura': np.clip(contraste[i], 0, 1),
                'homogeneidad_textura': np.clip(homog[i], 0, 1),
                'largo_cm': np.clip(largo[i], 10, 45),
                'ancho_cm': np.clip(ancho[i], 3, 15),
                'area_cm2': np.clip(area[i], 30, 400),
                'convexidad': np.clip(conv[i], 0.5, 1),
                'manchas_detectadas': int(np.clip(manchas[i], 0, 15)),
                'calidad': calidad
            })
    
    df = pd.DataFrame(records)
    # Mezclar bien
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


df = generar_dataset_vision(n=2000)

print(f'Dataset generado: {len(df)} piezas escaneadas')
print(f'\nDistribuciÃ³n de calidad:')
print(df['calidad'].value_counts().to_string())
print(f'\nPrimeras 3 filas:')
df.head(3)

### 2.1 â AnÃ¡lisis exploratorio visual

In [ ]:
fig = plt.figure(figsize=(15, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

palette = {'A': PESCATEAL, 'B': UTNCELESTE, 'C': '#E74C3C'}

# 1. DistribuciÃ³n de luminosidad por calidad
ax1 = fig.add_subplot(gs[0, 0])
for cal, color in palette.items():
    data = df[df['calidad'] == cal]['luminosidad']
    ax1.hist(data, bins=30, alpha=0.65, color=color, label=f'Grado {cal}', density=True)
ax1.set_xlabel('Luminosidad (0-255)')
ax1.set_ylabel('Densidad')
ax1.set_title('Luminosidad por grado')
ax1.legend(fontsize=9)

# 2. Homogeneidad de textura por calidad
ax2 = fig.add_subplot(gs[0, 1])
for cal, color in palette.items():
    data = df[df['calidad'] == cal]['homogeneidad_textura']
    ax2.hist(data, bins=30, alpha=0.65, color=color, label=f'Grado {cal}', density=True)
ax2.set_xlabel('Homogeneidad de textura')
ax2.set_title('Textura por grado')
ax2.legend(fontsize=9)

# 3. Manchas detectadas â boxplot
ax3 = fig.add_subplot(gs[0, 2])
data_manchas = [df[df['calidad'] == c]['manchas_detectadas'].values for c in ['A', 'B', 'C']]
bp = ax3.boxplot(data_manchas, labels=['Grado A', 'Grado B', 'Grado C'],
                 patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], palette.values()):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax3.set_ylabel('Manchas detectadas')
ax3.set_title('Defectos por grado de calidad')

# 4. Largo vs Ancho del filete (scatter)
ax4 = fig.add_subplot(gs[1, 0])
for cal, color in palette.items():
    sub = df[df['calidad'] == cal].sample(150)
    ax4.scatter(sub['largo_cm'], sub['ancho_cm'], c=color, alpha=0.5,
                label=f'Grado {cal}', s=25, edgecolors='none')
ax4.set_xlabel('Largo del filete (cm)')
ax4.set_ylabel('Ancho (cm)')
ax4.set_title('MorfologÃ­a del filete')
ax4.legend(fontsize=9)

# 5. SaturaciÃ³n vs Contraste
ax5 = fig.add_subplot(gs[1, 1])
for cal, color in palette.items():
    sub = df[df['calidad'] == cal].sample(150)
    ax5.scatter(sub['saturacion'], sub['contraste_textura'], c=color, alpha=0.5,
                label=f'Grado {cal}', s=25, edgecolors='none')
ax5.set_xlabel('SaturaciÃ³n de color')
ax5.set_ylabel('Contraste de textura')
ax5.set_title('SaturaciÃ³n vs Contraste')
ax5.legend(fontsize=9)

# 6. Mapa de calor de correlaciones
ax6 = fig.add_subplot(gs[1, 2])
features_num = ['luminosidad', 'saturacion', 'contraste_textura',
                 'homogeneidad_textura', 'largo_cm', 'manchas_detectadas']
corr = df[features_num].corr()
im = ax6.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax6.set_xticks(range(len(features_num)))
ax6.set_yticks(range(len(features_num)))
labels_short = ['Lum.', 'Sat.', 'Contr.', 'Homog.', 'Largo', 'Manchas']
ax6.set_xticklabels(labels_short, rotation=45, ha='right', fontsize=9)
ax6.set_yticklabels(labels_short, fontsize=9)
plt.colorbar(im, ax=ax6, fraction=0.046, pad=0.04)
ax6.set_title('Correlaciones entre features')

plt.suptitle('EDA â Dataset de VisiÃ³n Artificial en Planta Pesquera\n'
             '2.000 piezas de merluza hubbsi escaneadas en lÃ­nea de producciÃ³n',
             fontsize=13, fontweight='bold', color=UTNAZUL, y=1.01)
plt.show()

---
## Parte 3 â Clasificador de calidad

Entrenamos un modelo de Machine Learning que, a partir de las features extraÃ­das de la imagen,
predice automÃ¡ticamente el grado de calidad (A, B o C).

In [ ]:
# Features y target
feature_cols = [
    'color_R', 'color_G', 'color_B', 'luminosidad', 'saturacion',
    'contraste_textura', 'homogeneidad_textura',
    'largo_cm', 'ancho_cm', 'area_cm2', 'convexidad', 'manchas_detectadas'
]

X = df[feature_cols].values
y = df['calidad'].values

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Entrenamiento: {len(X_train)} piezas | Test: {len(X_test)} piezas')
print(f'DistribuciÃ³n train â A:{np.sum(y_train=="A")} B:{np.sum(y_train=="B")} C:{np.sum(y_train=="C")}')

# ââ Tres modelos ââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââââ
modelos = {
    'RegresiÃ³n LogÃ­stica': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=500, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=120, learning_rate=0.1, random_state=42))
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('\nââ ValidaciÃ³n cruzada 5-fold âââââââââââââââââââââââââââââââââ')
print(f'{"Modelo":<25} {"Accuracy CV":<15} {"Std"}')
print('-' * 50)

resultados = {}
for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=cv,
                             scoring='accuracy', n_jobs=-1)
    resultados[nombre] = scores
    print(f'{nombre:<25} {scores.mean():.4f}         Â± {scores.std():.4f}')

mejor_nombre = max(resultados, key=lambda k: resultados[k].mean())
print(f'\nMejor modelo en CV: {mejor_nombre}')

In [ ]:
# Entrenamos el mejor modelo con todos los datos de train y evaluamos en test
modelo_final = modelos[mejor_nombre]
modelo_final.fit(X_train, y_train)
y_pred = modelo_final.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'Accuracy en test: {acc:.4f} ({acc*100:.1f}%)')
print(f'\nReporte de clasificaciÃ³n:')
print(classification_report(y_test, y_pred, target_names=['Grado A', 'Grado B', 'Grado C']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusiÃ³n
cm = confusion_matrix(y_test, y_pred, labels=['A', 'B', 'C'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Grado A', 'Grado B', 'Grado C'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Matriz de ConfusiÃ³n â {mejor_nombre}\nAccuracy: {acc:.1%}',
                  fontweight='bold', color=UTNAZUL)

# Importancia de features (si Random Forest o GB)
clf_inner = modelo_final.named_steps['clf']
if hasattr(clf_inner, 'feature_importances_'):
    importances = clf_inner.feature_importances_
    indices = np.argsort(importances)[::-1]
    colors = [PESCATEAL if i == indices[0] else
              UTNCELESTE if i in indices[:4] else ARIELBLUE
              for i in range(len(feature_cols))]
    
    sorted_features = [feature_cols[i] for i in indices]
    sorted_imp = importances[indices]
    
    bars = axes[1].barh(range(len(sorted_features)), sorted_imp,
                        color=[colors[i] for i in indices], alpha=0.85)
    axes[1].set_yticks(range(len(sorted_features)))
    axes[1].set_yticklabels(sorted_features, fontsize=10)
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Importancia relativa')
    axes[1].set_title('Importancia de features\n(Â¿quÃ© mira el modelo para clasificar?)',
                      fontweight='bold', color=UTNAZUL)
    
    # Anotaciones
    for bar, imp in zip(bars, sorted_imp):
        axes[1].text(imp + 0.003, bar.get_y() + bar.get_height()/2,
                    f'{imp:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

if hasattr(clf_inner, 'feature_importances_'):
    top3 = [feature_cols[i] for i in indices[:3]]
    print('Interpretacion:')
    print(f'  Las 3 features mas importantes: {top3}')
    print('  El modelo prioriza estas caracteristicas vs evaluacion manual.')
else:
    print('Importancia de features no disponible para este modelo.')

---
## Parte 4 â Arquitectura CNN: cÃ³mo funcionan las redes para visiÃ³n

Los sistemas de visiÃ³n industrial modernos usan **Redes Neuronales Convolucionales (CNN)**.  
La ventaja sobre extraer features manualmente: la red *aprende* quÃ© features son relevantes.

Visualizamos la arquitectura conceptual sin necesidad de GPU:

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.set_xlim(0, 15)
ax.set_ylim(0, 5)
ax.axis('off')

# DefiniciÃ³n de capas CNN
capas = [
    {'x': 0.5, 'w': 1.2, 'h': 3.5, 'color': UTNAZUL,    'label': 'Imagen\n224Ã224Ã3', 'sub': '(RGB)'},
    {'x': 2.2, 'w': 0.6, 'h': 3.0, 'color': PESCATEAL,  'label': 'Conv2D\n32 filtros\n3Ã3', 'sub': 'ReLU'},
    {'x': 3.2, 'w': 0.5, 'h': 2.5, 'color': PESCATEAL,  'label': 'MaxPool\n2Ã2', 'sub': ''},
    {'x': 4.2, 'w': 0.6, 'h': 2.2, 'color': '#059669',  'label': 'Conv2D\n64 filtros\n3Ã3', 'sub': 'ReLU'},
    {'x': 5.2, 'w': 0.5, 'h': 1.8, 'color': '#059669',  'label': 'MaxPool\n2Ã2', 'sub': ''},
    {'x': 6.2, 'w': 0.6, 'h': 1.5, 'color': UTNCELESTE, 'label': 'Conv2D\n128 filtros\n3Ã3', 'sub': 'ReLU'},
    {'x': 7.5, 'w': 0.4, 'h': 1.2, 'color': UTNCELESTE, 'label': 'MaxPool', 'sub': ''},
    {'x': 8.5, 'w': 0.35,'h': 3.5, 'color': ARIELBLUE,  'label': 'Flatten\nâ vector', 'sub': ''},
    {'x': 9.7, 'w': 0.35,'h': 2.8, 'color': ARIELBLUE,  'label': 'Dense\n256 neu.', 'sub': 'Dropout 0.5'},
    {'x': 10.9,'w': 0.35,'h': 2.2, 'color': TECHGOLD,   'label': 'Dense\n64 neu.', 'sub': 'ReLU'},
    {'x': 12.0,'w': 0.5, 'h': 1.2, 'color': '#E74C3C',  'label': 'Output\n3 clases', 'sub': 'Softmax'},
]

for capa in capas:
    y0 = (5 - capa['h']) / 2
    rect = patches.FancyBboxPatch(
        (capa['x'], y0), capa['w'], capa['h'],
        boxstyle='round,pad=0.05',
        facecolor=capa['color'], edgecolor='white', linewidth=1.5, alpha=0.9
    )
    ax.add_patch(rect)
    ax.text(capa['x'] + capa['w']/2, y0 + capa['h']/2,
            capa['label'], ha='center', va='center',
            fontsize=8, color='white', fontweight='bold')
    if capa['sub']:
        ax.text(capa['x'] + capa['w']/2, y0 - 0.3,
                capa['sub'], ha='center', va='top', fontsize=7.5,
                color='gray', style='italic')

# Flechas entre capas
xs = [c['x'] + c['w'] for c in capas[:-1]]
x2s = [c['x'] for c in capas[1:]]
for x1, x2 in zip(xs, x2s):
    ax.annotate('', xy=(x2, 2.5), xytext=(x1, 2.5),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# Etiquetas de secciones
ax.text(3.5, 4.8, 'EXTRACCIÃN AUTOMÃTICA DE FEATURES', ha='center',
        fontsize=10, color=PESCATEAL, fontweight='bold')
ax.text(9.5, 4.8, 'CLASIFICACIÃN', ha='center',
        fontsize=10, color=ARIELBLUE, fontweight='bold')
ax.axvline(x=8.2, color='#CCC', linewidth=1, linestyle='--', ymax=0.85)

# Output labels
ax.text(13.2, 3.5, 'A â Premium', fontsize=9, color=PESCATEAL, fontweight='bold')
ax.text(13.2, 2.5, 'B â EstÃ¡ndar', fontsize=9, color=UTNCELESTE, fontweight='bold')
ax.text(13.2, 1.5, 'C â Descarte', fontsize=9, color='#E74C3C', fontweight='bold')

ax.set_title('Arquitectura CNN para clasificaciÃ³n de calidad de merluza\n'
             'Input: imagen 224Ã224 px  |  Output: probabilidad de cada grado (A/B/C)',
             fontsize=12, fontweight='bold', color=UTNAZUL)
plt.tight_layout()
plt.show()

print('Esta arquitectura puede entrenarse con ~5.000-10.000 imÃ¡genes etiquetadas.')
print('Transfer Learning (EfficientNet, MobileNet) reduce ese requerimiento a ~500-1.000 imÃ¡genes.')

---
## Parte 5 â SimulaciÃ³n de throughput industrial

In [ ]:
# ComparaciÃ³n manual vs IA en lÃ­nea de producciÃ³n
turnos_por_dia = 2
horas_turno = 8

metodos = {
    'Control manual\n(2 operarios)': {
        'piezas_por_min': 75,
        'precision_A': 0.82,   # detecta 82% de los A correctamente
        'precision_C': 0.70,   # detecta 70% de los descarte correctamente
        'costo_diario_usd': 280,
        'color': '#E74C3C'
    },
    'IA + cÃ¡mara\n(1 supervisor)': {
        'piezas_por_min': 280,
        'precision_A': 0.96,
        'precision_C': 0.93,
        'costo_diario_usd': 90,   # amortizaciÃ³n equipo + supervisor
        'color': PESCATEAL
    }
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

nombres = list(metodos.keys())
colores = [m['color'] for m in metodos.values()]

# Throughput diario
throughput = [m['piezas_por_min'] * 60 * horas_turno * turnos_por_dia / 1000
              for m in metodos.values()]
bars1 = axes[0].bar(nombres, throughput, color=colores, alpha=0.85, width=0.5)
axes[0].set_ylabel('Miles de piezas/dÃ­a')
axes[0].set_title('Capacidad de procesamiento', fontweight='bold')
for bar, val in zip(bars1, throughput):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.0f}K', ha='center', fontweight='bold', fontsize=12)

# PrecisiÃ³n en detecciÃ³n de grado A
prec_A = [m['precision_A']*100 for m in metodos.values()]
bars2 = axes[1].bar(nombres, prec_A, color=colores, alpha=0.85, width=0.5)
axes[1].set_ylabel('PrecisiÃ³n (%)')
axes[1].set_ylim(60, 100)
axes[1].set_title('PrecisiÃ³n clasificaciÃ³n Grado A', fontweight='bold')
for bar, val in zip(bars2, prec_A):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.0f}%', ha='center', fontweight='bold', fontsize=12)

# Costo diario
costo = [m['costo_diario_usd'] for m in metodos.values()]
bars3 = axes[2].bar(nombres, costo, color=colores, alpha=0.85, width=0.5)
axes[2].set_ylabel('USD / dÃ­a')
axes[2].set_title('Costo operativo diario', fontweight='bold')
for bar, val in zip(bars3, costo):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'USD {val}', ha='center', fontweight='bold', fontsize=12)

plt.suptitle('Control Manual vs VisiÃ³n Artificial â ComparaciÃ³n en lÃ­nea de producciÃ³n pesquera',
             fontsize=12, fontweight='bold', color=UTNAZUL)
plt.tight_layout()
plt.show()

# ROI estimado
aumento_throughput = throughput[1] / throughput[0]
ahorro_diario = costo[0] - costo[1]
print(f'\nResumen del impacto econÃ³mico:')
print(f'  Aumento de throughput:       {aumento_throughput:.1f}x mÃ¡s piezas procesadas/dÃ­a')
print(f'  Ahorro operativo diario:     USD {ahorro_diario}')
print(f'  Ahorro anual (250 dÃ­as):     USD {ahorro_diario * 250:,}')
print(f'  Payback de equipo tÃ­pico:    ~12-18 meses (equipo USD 60.000-80.000)')

---
## Parte 6 â PredicciÃ³n en tiempo real: el sistema en producciÃ³n

In [ ]:
# Simulamos 5 piezas que llegan a la cÃ¡mara y el sistema clasifica en tiempo real
piezas_nuevas = pd.DataFrame([
    {'desc': 'Filete grande, muy blanco, sin defectos',
     'color_R': 205, 'color_G': 196, 'color_B': 188, 'luminosidad': 195,
     'saturacion': 0.10, 'contraste_textura': 0.07, 'homogeneidad_textura': 0.92,
     'largo_cm': 31, 'ancho_cm': 9.2, 'area_cm2': 205, 'convexidad': 0.97,
     'manchas_detectadas': 0},
    {'desc': 'Filete mediano, algunas manchas menores',
     'color_R': 180, 'color_G': 170, 'color_B': 158, 'luminosidad': 170,
     'saturacion': 0.25, 'contraste_textura': 0.20, 'homogeneidad_textura': 0.72,
     'largo_cm': 23, 'ancho_cm': 7.0, 'area_cm2': 118, 'convexidad': 0.88,
     'manchas_detectadas': 2},
    {'desc': 'Pieza con decoloraciÃ³n marcada y borde irregular',
     'color_R': 155, 'color_G': 138, 'color_B': 120, 'luminosidad': 138,
     'saturacion': 0.48, 'contraste_textura': 0.40, 'homogeneidad_textura': 0.55,
     'largo_cm': 17, 'ancho_cm': 5.5, 'area_cm2': 75, 'convexidad': 0.72,
     'manchas_detectadas': 6},
    {'desc': 'Filete premium, blanco uniforme, talla Ã³ptima',
     'color_R': 208, 'color_G': 200, 'color_B': 192, 'luminosidad': 200,
     'saturacion': 0.08, 'contraste_textura': 0.06, 'homogeneidad_textura': 0.94,
     'largo_cm': 34, 'ancho_cm': 10, 'area_cm2': 245, 'convexidad': 0.98,
     'manchas_detectadas': 0},
    {'desc': 'Pieza pequeÃ±a, manchas mÃºltiples, textura irregular',
     'color_R': 160, 'color_G': 145, 'color_B': 128, 'luminosidad': 145,
     'saturacion': 0.42, 'contraste_textura': 0.38, 'homogeneidad_textura': 0.58,
     'largo_cm': 14, 'ancho_cm': 4.5, 'area_cm2': 55, 'convexidad': 0.76,
     'manchas_detectadas': 7},
])

X_nuevas = piezas_nuevas[feature_cols].values
predicciones = modelo_final.predict(X_nuevas)
probabilidades = modelo_final.predict_proba(X_nuevas)
clases = modelo_final.classes_

grade_colors = {'A': PESCATEAL, 'B': UTNCELESTE, 'C': '#E74C3C'}
grade_labels = {'A': 'â PREMIUM', 'B': '~ ESTÃNDAR', 'C': 'â DESCARTE'}

print('=' * 70)
print('SISTEMA DE CLASIFICACIÃN EN TIEMPO REAL â Resultados')
print('=' * 70)

for i, (_, pieza) in enumerate(piezas_nuevas.iterrows()):
    pred = predicciones[i]
    probs = probabilidades[i]
    probs_dict = dict(zip(clases, probs))
    print(f'\nPieza #{i+1}: {pieza["desc"]}')
    print(f'  ClasificaciÃ³n â Grado {pred} ({grade_labels[pred]})')
    print(f'  Confianza:  A={probs_dict["A"]:.1%}  B={probs_dict["B"]:.1%}  C={probs_dict["C"]:.1%}')

print('\n' + '=' * 70)
print('Tiempo de procesamiento simulado: < 5 ms por pieza')
print('En velocidad de lÃ­nea: 280 piezas/min â 4.7 piezas/segundo')

---
## Parte 7 â ReflexiÃ³n y pasos siguientes

### Aplicaciones adicionales de visiÃ³n artificial en el sector pesquero

| AplicaciÃ³n | Input (imagen) | Output del modelo |
|-----------|---------------|-------------------|
| **ClasificaciÃ³n por especie** | Imagen del pez entero | Merluza / Polaca / CastaÃ±eta / ... |
| **MediciÃ³n automÃ¡tica de talla** | Imagen lateral | Largo (cm) con error < 2 mm |
| **DetecciÃ³n de parÃ¡sitos** | Imagen de filete con luz transmitted | Presencia/ausencia y zona |
| **Control de porcionado** | Imagen de porciÃ³n | Peso estimado (g) sin balanza |
| **EvaluaciÃ³n de frescura** | Imagen de ojos + branquias | Score 1-5 (muy fresco â deteriorado) |
| **DetecciÃ³n de espinas** | Imagen de rayos X del filete | Mapa de espinas residuales |

### Â¿QuÃ© necesita una planta para implementar esto?

1. **Hardware:** cÃ¡mara industrial (basler/cognex) + PC con GPU (opcional) + iluminaciÃ³n controlada
2. **Datos:** 500-5.000 imÃ¡genes etiquetadas por personal del sector (con Transfer Learning: 500 son suficientes)
3. **Software:** Python + OpenCV + PyTorch/TensorFlow (todo gratuito y open-source)
4. **IntegraciÃ³n:** API con PLC o SCADA de la lÃ­nea para activar deflectores fÃ­sicos
5. **ValidaciÃ³n:** auditorÃ­a por biÃ³logos durante el primer mes de operaciÃ³n

### ConexiÃ³n con el resto del curso
- **Clase 4:** los datos AIS y de sensor son anÃ¡logos a las features de imagen â pipelines similares
- **Clase 6:** el clasificador de calidad que construimos hoy usa exactamente el mismo flujo que el predictor de zonas de pesca
- **Clase 8:** el sistema de visiÃ³n puede integrarse al dashboard de gestiÃ³n de flota para trazabilidad completa

In [ ]:
# Resumen final
print('RESUMEN DEL NOTEBOOK â Clase 7: VisiÃ³n Artificial en Plantas Pesqueras')
print('=' * 65)
print()
print('Dataset: 2.000 piezas de merluza hubbsi con 12 features de imagen')
print(f'Mejor modelo: {mejor_nombre}')
print(f'Accuracy en test: {acc:.1%}')
print()
print('Lo que aprendimos:')
print('  1. Las features de imagen (color, textura, morfologÃ­a) discriminan calidad')
print('  2. Un Random Forest/GB clasifica bien con features bien diseÃ±adas')
print('  3. Las CNNs aprenden esas features automÃ¡ticamente desde pÃ­xeles crudos')
print('  4. El impacto econÃ³mico es concreto: 3.7x mÃ¡s throughput, 68% menos costo op.')
print()
print('Materiales: github.com/PesquerosEnIA/curso-ia-produccion-pesquera')